### Problem Statement 4A

4A. Implement the CBOW for document classification using deep learning. 

Steps (for Implementing CBOW for Document Classification):
1. Data Preparation: You need a labeled dataset for document classification. You can use datasets like the IMDb movie reviews dataset (for sentiment classification) or 20 Newsgroups dataset (for topic classification).
Load and Preprocess the data:
- from tensorflow.keras.datasets import imdb # For loading
- Pad sequences to ensure uniform input length
- Convert the IMDB dataset to a list of sentences (each sentence is a list of word)
- Tokenize the text into words.
- Lowercase the text and remove stop words.
- Create a vocabulary from the training data.
2. Define CBOW Model: CBOW learns to predict a target word based on surrounding context words. For document classification, you need to modify the CBOW model to work with document-level contexts. In CBOW, the context consists of words surrounding the target word, but for documents, you will use the embeddings from all the words in the document.
3. Model Architecture:
- Embedding Layer: First, use the CBOW architecture to learn word embeddings from the text corpus.
- Document Representation: After learning the word embeddings, represent a document by averaging the embeddings of all words in the document.
- Fully Connected Layer: Use this document representation as input to a neural network (e.g., a dense layer) for classification.
- Train word2vec model with CBOW (sg=0 for CBOW)
- Convert words to embeddings
- Build the feed forward neural network for document classification
4. Evaluate the Model on test set using loss and accuracy.

In [ ]:
# Step 1: Import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from gensim.models import Word2Vec
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from sklearn.model_selection import train_test_split

In [ ]:
# Step 2: Load dataset
data = pd.read_csv(
    "../LP IV Updated Datasets/CBOW/IMDB Dataset.csv"
).head(3000)

print(data.head(2))

In [ ]:
# Step 3: Clean and tokenize reviews
sentences = []

for review in data['review']:
    words = [
        w.lower() for w in review.split()
        if w.isalpha() and len(w) > 2
    ]
    sentences.append(words)

# Convert sentiment into numbers
y = (data['sentiment'] == 'positive').astype(int).values

print("Total reviews:", len(sentences))

In [ ]:
# Step 4: Train CBOW model
cbow_model = Word2Vec(
    sentences,
    vector_size=100,
    window=5,
    min_count=2,
    sg=0,
    epochs=10
)

In [ ]:
# Step 5: Convert reviews into document vectors
def get_document_embedding(words):

    vectors = [
        cbow_model.wv[w]
        for w in words
        if w in cbow_model.wv
    ]

    if len(vectors) == 0:
        return np.zeros(100)

    return np.mean(vectors, axis=0)


X = np.array([
    get_document_embedding(words)
    for words in sentences
])

print("Feature matrix:", X.shape)

In [ ]:
# Step 6: Split dataset
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

In [ ]:
# Step 7: Build neural network
model = Sequential()

model.add(Dense(32, activation='relu', input_shape=(100,)))
model.add(Dense(16, activation='relu'))
model.add(Dense(1, activation='sigmoid'))

# Compile model
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

In [ ]:
# Step 8: Train model
history = model.fit(
    X_train,
    y_train,
    epochs=10,
    batch_size=32,
    validation_split=0.1
)

In [ ]:
# Step 9: Evaluate model
loss, accuracy = model.evaluate(X_test, y_test)

print("Test Loss:", loss)
print("Test Accuracy:", accuracy)

In [ ]:
# Step 10: Plot accuracy and loss
plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'])
plt.plot(history.history['val_accuracy'])
plt.title('Accuracy')
plt.legend(['Train', 'Validation'])

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])
plt.title('Loss')
plt.legend(['Train', 'Validation'])

plt.show()